# RAG dev - parsing the handbook chapters

Scratch notebook for checking `retrieval/parse.py` by eye on the real FAA
Powerplant Handbook chapters. All the logic lives in
`src/industrial_agent/retrieval/`; this only calls it and prints.

## Setup

In [ ]:
import os, sys

if not os.path.exists("repo"):
    !git clone -q -b feature/maintenance-docs-rag https://github.com/Cambie26/industrial_agent.git repo
%pip install -q -e "./repo[rag]"

sys.path.insert(0, os.path.abspath("repo/src"))
sys.path.insert(0, os.path.abspath("repo"))

## Corpus

Upload one PDF per chapter to `repo/data/raw/`, named with the chapter number
(`ch10.pdf`). `data/` is gitignored, so they won't be committed. To read them
from Drive instead, mount it and point `RAW_DIR` at the folder.

In [ ]:
from collections import Counter
from pathlib import Path

from industrial_agent.retrieval import parse

RAW_DIR = Path("repo/data/raw")  # or Path("/content/drive/MyDrive/faa_powerplant")
RAW_DIR.mkdir(parents=True, exist_ok=True)

# from google.colab import drive; drive.mount("/content/drive")

pdf_paths = parse.chapter_pdfs(RAW_DIR)
for pdf_path in pdf_paths:
    print(f"chapter {parse.chapter_number(pdf_path):>2}: {pdf_path.name}")

## Parse one chapter

In [ ]:
CHAPTER = 10
pdf_path = next(path for path in pdf_paths if parse.chapter_number(path) == CHAPTER)

blocks = parse.parse_pdf(pdf_path)
pages = {block.page for block in blocks}
print(f"{pdf_path.name}: {len(pages)} pages, {len(blocks)} blocks")
print(Counter(block.kind for block in blocks))

### Contents

Chunking takes its section boundaries from this. If it raises, the chapter
PDF has no bookmarks and the contents page will need parsing instead.

In [ ]:
for level, title, page in parse.table_of_contents(pdf_path):
    print(f"{'  ' * (level - 1)}{title}  (p.{page})")

### Reading order on one page

Check the two columns come out left then right, not interleaved.

In [ ]:
PAGE = 3
for block in blocks:
    if block.page == PAGE:
        print(f"[{block.kind}] {block.text[:200]}\n")

### Tables, as markdown

In [ ]:
from IPython.display import Markdown, display

tables = [block for block in blocks if block.kind == "table"]
print(f"{len(tables)} tables")
for table in tables[:5]:
    display(Markdown(f"**page {table.page}**\n\n{table.text}"))

### Figure captions

In [ ]:
for caption in (block for block in blocks if block.kind == "caption"):
    print(f"p.{caption.page:>3}  {caption.text}")

## Naive vs table-aware

The naive parse is the ablation baseline: plain page text, tables flattened
into the prose. Compare the same page both ways.

In [ ]:
naive_blocks = parse.parse_pdf(pdf_path, table_aware=False)
table_page = tables[0].page if tables else PAGE

print("=== naive ===")
print(next(block.text for block in naive_blocks if block.page == table_page)[:1500])
print("\n=== table-aware ===")
print(parse.to_markdown([block for block in blocks if block.page == table_page])[:1500])

## Save the parse for reading in full

In [ ]:
PARSED_DIR = Path("repo/data/parsed")
PARSED_DIR.mkdir(parents=True, exist_ok=True)

markdown_path = PARSED_DIR / f"{pdf_path.stem}.md"
markdown_path.write_text(parse.to_markdown(blocks))
print(f"wrote {markdown_path} ({markdown_path.stat().st_size / 1024:.0f} kB)")

## Whole corpus

In [ ]:
corpus_blocks = parse.parse_corpus(RAW_DIR)
by_chapter = Counter((block.chapter, block.kind) for block in corpus_blocks)
for chapter in sorted({block.chapter for block in corpus_blocks}):
    counts = {kind: by_chapter[(chapter, kind)] for kind in ("text", "table", "caption")}
    print(f"chapter {chapter:>2}: {counts}")